In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# =========================================================
# CONFIGURACIÓN GENERAL
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.csv"

N_SAMPLE = 100          # tamaño del bootstrap balanceado por iteración
N_ITER = 500            # iteraciones bootstrap por signo
K_FOLDS = 5             # folds CV dentro del bootstrap
BASE_SEED = 42

CARPETA_OUT_BASE = Path("../5_results/bootstrap_cv_models_v2")
CARPETA_OUT_BASE.mkdir(parents=True, exist_ok=True)

# =========================================================
# HIPERPARÁMETROS (razonables + reproducibles)
# ---------------------------------------------------------
# NOTAS:
# - RF/DT no requieren escalado.
# - KNN/LR/SVM sí se benefician de escalado.
# - En SVM usamos kernel="linear" y probability=True para poder calcular AUC.
# =========================================================
RF_PARAMS = dict(
    n_estimators=200,        # cantidad de árboles
    max_depth=None,          # None = crecer hasta que no pueda
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=2,
    random_state=BASE_SEED
)

DT_PARAMS = dict(
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=BASE_SEED
)

KNN_PARAMS = dict(
    n_neighbors=50,          # k (alto para suavizar ruido)
    weights="uniform",
    metric="euclidean",
    n_jobs=1
)

LR_PARAMS = dict(
    solver="liblinear",      # estable para binario (one-vs-rest)
    C=1.0,                   # regularización (↓C = más regularización)
    max_iter=2000,
    random_state=BASE_SEED
)

SVM_PARAMS = dict(
    kernel="linear",         # lineal (rápido y razonable con one-hot)
    C=1.0,
    probability=True,        # necesario para ROC/PR AUC
    random_state=BASE_SEED
)

MODELOS = {
    "lr": Pipeline(steps=[
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("lr", LogisticRegression(**LR_PARAMS))
    ]),
    "svm": Pipeline(steps=[
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("svm", SVC(**SVM_PARAMS))
    ]),
    "rf": RandomForestClassifier(**RF_PARAMS),
    "dt": DecisionTreeClassifier(**DT_PARAMS),
    "knn": Pipeline(steps=[
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("knn", KNeighborsClassifier(**KNN_PARAMS))
    ]),
}

# =========================================================
# 1) CARGA DATASET
# =========================================================
print("\n--- BOOTSTRAP + CV | LR + SVM + RF + DT + KNN | TODOS LOS SIGNOS ---")
df = pd.read_csv(RUTA_DATASET, low_memory=False)
print("Dataset cargado:", df.shape)

# Targets: columnas one-vs-rest
cols_signo = [c for c in df.columns if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas target tipo 'signo_zodiacal_*' en el dataset.")

# Features: todo excepto columnas target y la columna multiclase (si existe)
drop_cols = cols_signo.copy()
if "SIGNO_ZODIACAL" in df.columns:
    drop_cols.append("SIGNO_ZODIACAL")

X_all = df.drop(columns=drop_cols).copy()

# Chequeo: todo debe ser numérico
obj_cols = X_all.select_dtypes(include=["object"]).columns.tolist()
if obj_cols:
    raise ValueError(f"Hay columnas no numéricas en X: {obj_cols}. Revisa la binarización.")

print("Total features:", X_all.shape[1])
print("Total targets (signos):", len(cols_signo))

# CV dentro de cada sample n=100
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# FUNCIÓN PRINCIPAL
# =========================================================
def run_model(nombre_modelo, modelo):
    print(f"\n==============================")
    print(f"🚀 MODELO: {nombre_modelo.upper()}")
    print(f"==============================")

    resultados = []

    for signo in cols_signo:
        print(f"\n🔮 Evaluando signo: {signo}")
        y_all = df[signo].astype(int)

        pos_idx = df.index[y_all == 1].to_numpy()
        neg_idx = df.index[y_all == 0].to_numpy()

        if len(pos_idx) == 0 or len(neg_idx) == 0:
            print(f"⚠️ Saltando {signo} (sin suficientes datos)")
            continue

        n_pos = N_SAMPLE // 2
        n_neg = N_SAMPLE - n_pos

        for it in range(N_ITER):
            seed_it = BASE_SEED + it
            rng_it = np.random.default_rng(seed_it)

            # -------------------------------------------------
            # 1) Bootstrap balanceado (n=100)
            # -------------------------------------------------
            sample_pos = rng_it.choice(pos_idx, size=n_pos, replace=True)
            sample_neg = rng_it.choice(neg_idx, size=n_neg, replace=True)
            sample_idx = np.concatenate([sample_pos, sample_neg])
            rng_it.shuffle(sample_idx)

            X = X_all.loc[sample_idx].reset_index(drop=True)
            y = y_all.loc[sample_idx].reset_index(drop=True)

            pos_rate = float(y.mean())

            fold_metrics = []
            fold_conf = []
            nan_roc = 0
            nan_pr = 0

            # -------------------------------------------------
            # 2) CV estratificada (k=5)
            # -------------------------------------------------
            for tr_idx, te_idx in skf.split(X, y):
                X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
                y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

                m = clone(modelo)
                m.fit(X_tr, y_tr)

                y_pred = m.predict(X_te)

                tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0, 1]).ravel()
                fold_conf.append([tp, fp, tn, fn])

                acc = accuracy_score(y_te, y_pred)
                prec = precision_score(y_te, y_pred, zero_division=0)
                rec = recall_score(y_te, y_pred, zero_division=0)
                f1 = f1_score(y_te, y_pred, zero_division=0)

                y_proba = None
                if hasattr(m, "predict_proba"):
                    y_proba = m.predict_proba(X_te)[:, 1]

                if y_proba is not None:
                    try:
                        roc = roc_auc_score(y_te, y_proba)
                    except ValueError:
                        roc = np.nan
                        nan_roc += 1

                    try:
                        pr = average_precision_score(y_te, y_proba)
                    except ValueError:
                        pr = np.nan
                        nan_pr += 1
                else:
                    roc = np.nan
                    pr = np.nan
                    nan_roc += 1
                    nan_pr += 1

                fold_metrics.append([acc, prec, rec, f1, roc, pr])

            fold_metrics = np.array(fold_metrics, dtype=float)
            fold_conf = np.array(fold_conf, dtype=float)

            resultados.append({
                "modelo": nombre_modelo,
                "signo": signo,
                "iter": it + 1,
                "pos_rate_sample": pos_rate,

                "accuracy": np.nanmean(fold_metrics[:, 0]),
                "precision": np.nanmean(fold_metrics[:, 1]),
                "recall": np.nanmean(fold_metrics[:, 2]),
                "f1": np.nanmean(fold_metrics[:, 3]),
                "roc_auc": np.nanmean(fold_metrics[:, 4]),
                "pr_auc": np.nanmean(fold_metrics[:, 5]),

                "tp_mean": np.mean(fold_conf[:, 0]),
                "fp_mean": np.mean(fold_conf[:, 1]),
                "tn_mean": np.mean(fold_conf[:, 2]),
                "fn_mean": np.mean(fold_conf[:, 3]),

                "nan_roc_folds": nan_roc,
                "nan_pr_folds": nan_pr,
            })

            if (it + 1) % 100 == 0:
                print(f"   Iter {it+1}/{N_ITER} completada")

    return pd.DataFrame(resultados)

# =========================================================
# 2) EJECUTAR Y GUARDAR
# =========================================================
for nombre_modelo, modelo in MODELOS.items():
    df_detalle = run_model(nombre_modelo, modelo)

    carpeta_modelo = CARPETA_OUT_BASE / f"{nombre_modelo}_bootstrap_cv"
    carpeta_modelo.mkdir(parents=True, exist_ok=True)

    detalle_path = carpeta_modelo / f"{nombre_modelo}_detalle.csv"
    resumen_path = carpeta_modelo / f"{nombre_modelo}_resumen.csv"

    df_detalle.to_csv(detalle_path, index=False)

    cols_metricas = [
        "pos_rate_sample",
        "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc",
        "tp_mean", "fp_mean", "tn_mean", "fn_mean",
        "nan_roc_folds", "nan_pr_folds"
    ]

    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)

    print(f"\n✅ Guardado {nombre_modelo.upper()}")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)

print("\n✅ TODO COMPLETADO (LR + SVM + RF + DT + KNN)")


--- BOOTSTRAP + CV | RF + DT + KNN | TODOS LOS SIGNOS ---
Dataset cargado: (5808417, 299)

🚀 MODELO: RF

🔮 Evaluando signo: signo_zodiacal_acuario
   Iter 100/500 completada
   Iter 200/500 completada
   Iter 300/500 completada
   Iter 400/500 completada
   Iter 500/500 completada

🔮 Evaluando signo: signo_zodiacal_aries
   Iter 100/500 completada
   Iter 200/500 completada
   Iter 300/500 completada
   Iter 400/500 completada
   Iter 500/500 completada

🔮 Evaluando signo: signo_zodiacal_capricornio
   Iter 100/500 completada
   Iter 200/500 completada
   Iter 300/500 completada
   Iter 400/500 completada
   Iter 500/500 completada

🔮 Evaluando signo: signo_zodiacal_cancer
   Iter 100/500 completada
   Iter 200/500 completada
   Iter 300/500 completada
   Iter 400/500 completada
   Iter 500/500 completada

🔮 Evaluando signo: signo_zodiacal_escorpio
   Iter 100/500 completada
   Iter 200/500 completada
   Iter 300/500 completada
   Iter 400/500 completada
   Iter 500/500 completada

🔮 

# Training LR

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

# =========================================================
# CONFIGURACIÓN GENERAL
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_binarizado.parquet"

cols_signo = [
    'signo_zodiacal_acuario',
    'signo_zodiacal_aries',
    'signo_zodiacal_capricornio',
    'signo_zodiacal_cancer',
    'signo_zodiacal_escorpio',
    'signo_zodiacal_geminis',
    'signo_zodiacal_leo',
    'signo_zodiacal_libra',
    'signo_zodiacal_piscis',
    'signo_zodiacal_sagitario',
    'signo_zodiacal_tauro',
    'signo_zodiacal_virgo'
]

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42

# Carpeta salida (solo LR)
CARPETA_OUT_BASE = Path("../5_results/bootstrap_cv_models")
CARPETA_OUT_BASE.mkdir(parents=True, exist_ok=True)

# =========================================================
# HIPERPARÁMETROS LR (ajusta si quieres)
# =========================================================
# Nota:
# - SAGA soporta datasets grandes, L1/L2/elasticnet (usamos L2 por defecto).
# - class_weight=None porque tú ya balanceas vía bootstrap 50/50.
# - max_iter alto por seguridad.
LR_PARAMS = dict(
    C=1.0,
    penalty="l2",
    solver="saga",
    max_iter=5000,
    n_jobs=2,
    random_state=BASE_SEED
)

# Pipeline (es importante escalar para LR)
LR_MODEL = Pipeline(steps=[
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("lr", LogisticRegression(**LR_PARAMS))
])

# =========================================================
# 1) CARGA DATASET
# =========================================================
print("\n--- LOGISTIC REGRESSION | BOOTSTRAP + CV | TODOS LOS SIGNOS ---")
df = pd.read_parquet(RUTA_DATASET)
print("Dataset cargado:", df.shape)

# Features: diagnósticos
cols_diag = [
    c for c in df.columns
    if c.startswith("diagnostico1_")
    or c.startswith("diagnostico2_")
    or c.startswith("diagnostico3_")
]
if not cols_diag:
    raise ValueError("No se encontraron columnas de diagnósticos.")

X_all = df[cols_diag].copy()

# CV dentro de cada sample n=100
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# 2) LOOP PRINCIPAL
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\n🔮 Evaluando signo: {signo}")
    y_all = df[signo].astype(int)

    pos_idx = df.index[y_all == 1].to_numpy()
    neg_idx = df.index[y_all == 0].to_numpy()

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"⚠️ Saltando {signo} (sin suficientes datos)")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    for it in range(N_ITER):
        seed_it = BASE_SEED + it
        rng_it = np.random.default_rng(seed_it)

        # -------------------------------------------------
        # 1) Bootstrap balanceado (n=100)
        # -------------------------------------------------
        sample_pos = rng_it.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng_it.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng_it.shuffle(sample_idx)

        X = X_all.loc[sample_idx].reset_index(drop=True)
        y = y_all.loc[sample_idx].reset_index(drop=True)

        # chequeo de balance real (debería ser ~0.5)
        pos_rate = float(y.mean())

        fold_metrics = []
        fold_conf = []
        nan_roc = 0
        nan_pr = 0

        # -------------------------------------------------
        # 2) CV estratificada (k=5)
        # -------------------------------------------------
        for tr_idx, te_idx in skf.split(X, y):
            X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
            y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

            m = clone(LR_MODEL)
            m.fit(X_tr, y_tr)

            y_pred = m.predict(X_te)

            # Confusion matrix robusta (labels=[0,1] fuerza 2x2)
            tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0, 1]).ravel()
            fold_conf.append([tp, fp, tn, fn])

            # Métricas clásicas
            acc = accuracy_score(y_te, y_pred)
            prec = precision_score(y_te, y_pred, zero_division=0)
            rec = recall_score(y_te, y_pred, zero_division=0)
            f1 = f1_score(y_te, y_pred, zero_division=0)

            # Probabilidades para AUC
            y_proba = None
            if hasattr(m, "predict_proba"):
                y_proba = m.predict_proba(X_te)[:, 1]

            if y_proba is not None:
                try:
                    roc = roc_auc_score(y_te, y_proba)
                except ValueError:
                    roc = np.nan
                    nan_roc += 1

                try:
                    pr = average_precision_score(y_te, y_proba)
                except ValueError:
                    pr = np.nan
                    nan_pr += 1
            else:
                roc = np.nan
                pr = np.nan
                nan_roc += 1
                nan_pr += 1

            fold_metrics.append([acc, prec, rec, f1, roc, pr])

        fold_metrics = np.array(fold_metrics, dtype=float)
        fold_conf = np.array(fold_conf, dtype=float)

        # Promedios por iteración (promedio sobre folds)
        resultados.append({
            "modelo": "lr",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_sample": pos_rate,

            "accuracy": np.nanmean(fold_metrics[:, 0]),
            "precision": np.nanmean(fold_metrics[:, 1]),
            "recall": np.nanmean(fold_metrics[:, 2]),
            "f1": np.nanmean(fold_metrics[:, 3]),
            "roc_auc": np.nanmean(fold_metrics[:, 4]),
            "pr_auc": np.nanmean(fold_metrics[:, 5]),

            "tp_mean": np.mean(fold_conf[:, 0]),
            "fp_mean": np.mean(fold_conf[:, 1]),
            "tn_mean": np.mean(fold_conf[:, 2]),
            "fn_mean": np.mean(fold_conf[:, 3]),

            "nan_roc_folds": nan_roc,
            "nan_pr_folds": nan_pr,
        })

        if (it + 1) % 100 == 0:
            print(f"   Iter {it+1}/{N_ITER} completada")

# =========================================================
# 3) GUARDAR RESULTADOS
# =========================================================
df_detalle = pd.DataFrame(resultados)

carpeta_modelo = CARPETA_OUT_BASE / "lr_bootstrap_cv"
carpeta_modelo.mkdir(parents=True, exist_ok=True)

detalle_path = carpeta_modelo / "lr_detalle.csv"
resumen_path = carpeta_modelo / "lr_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_sample",
    "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc",
    "tp_mean", "fp_mean", "tn_mean", "fn_mean",
    "nan_roc_folds", "nan_pr_folds"
]

df_resumen = (
    df_detalle
    .groupby("signo")[cols_metricas]
    .agg(["mean", "std"])
)
df_resumen.to_csv(resumen_path)

print("\n✅ LOGISTIC REGRESSION COMPLETADO")
print("Detalle:", detalle_path)
print("Resumen:", resumen_path)


--- LOGISTIC REGRESSION | BOOTSTRAP + CV | TODOS LOS SIGNOS ---
Dataset cargado: (5808417, 299)

🔮 Evaluando signo: signo_zodiacal_acuario
   Iter 100/500 completada
   Iter 200/500 completada
   Iter 300/500 completada
   Iter 400/500 completada
   Iter 500/500 completada

🔮 Evaluando signo: signo_zodiacal_aries
   Iter 100/500 completada
   Iter 200/500 completada
   Iter 300/500 completada
   Iter 400/500 completada
   Iter 500/500 completada

🔮 Evaluando signo: signo_zodiacal_capricornio
   Iter 100/500 completada
   Iter 200/500 completada
   Iter 300/500 completada
   Iter 400/500 completada
   Iter 500/500 completada

🔮 Evaluando signo: signo_zodiacal_cancer
   Iter 100/500 completada
   Iter 200/500 completada
   Iter 300/500 completada
   Iter 400/500 completada
   Iter 500/500 completada

🔮 Evaluando signo: signo_zodiacal_escorpio
   Iter 100/500 completada
   Iter 200/500 completada
   Iter 300/500 completada
   Iter 400/500 completada
   Iter 500/500 completada

🔮 Evaluand

# Mostrar Resultados

In [1]:
import pandas as pd
from pathlib import Path

# =====================================================
# RUTAS DE LOS RESÚMENES
# =====================================================
BASE_PATH = Path("../5_results/bootstrap_cv_models")

resumenes = {
    "Random Forest": BASE_PATH / "rf_bootstrap_cv" / "rf_resumen.csv",
    "Árbol de Decisión": BASE_PATH / "dt_bootstrap_cv" / "dt_resumen.csv",
    "KNN": BASE_PATH / "knn_bootstrap_cv" / "knn_resumen.csv",
    "Regresión Logística": BASE_PATH / "lr_bootstrap_cv" / "lr_resumen.csv",
}

medias_modelos = {}

# =====================================================
# CARGA Y PROCESAMIENTO
# =====================================================
for nombre_modelo, ruta in resumenes.items():
    print(f"\n📊 {nombre_modelo}")
    print("-" * 50)

    df_resumen = pd.read_csv(ruta, header=[0, 1], index_col=0)

    # Extraer solo la media
    df_medias = df_resumen.xs("mean", axis=1, level=1)

    # Redondear
    df_medias = df_medias.round(3)
    

    medias_modelos[nombre_modelo] = df_medias

    display(df_medias.style.format("{:.3f}"))



📊 Random Forest
--------------------------------------------------


FileNotFoundError: [Errno 2] No such file or directory: '..\\5_results\\bootstrap_cv_models\\rf_bootstrap_cv\\rf_resumen.csv'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =====================================================
# CONFIGURACIÓN
# =====================================================
BASE_PATH = Path("../5_results/bootstrap_cv_models")

MODELOS = {
    "Random Forest": BASE_PATH / "rf_bootstrap_cv" / "rf_detalle.csv",
    "Árbol de Decisión": BASE_PATH / "dt_bootstrap_cv" / "dt_detalle.csv",
    "KNN": BASE_PATH / "knn_bootstrap_cv" / "knn_detalle.csv",
    "Regresión Logística": BASE_PATH / "lr_bootstrap_cv" / "lr_detalle.csv",
}

signos = [
    'signo_zodiacal_acuario',
    'signo_zodiacal_aries',
    'signo_zodiacal_capricornio',
    'signo_zodiacal_cancer',
    'signo_zodiacal_escorpio',
    'signo_zodiacal_geminis',
    'signo_zodiacal_leo',
    'signo_zodiacal_libra',
    'signo_zodiacal_piscis',
    'signo_zodiacal_sagitario',
    'signo_zodiacal_tauro',
    'signo_zodiacal_virgo'
]

CARPETA_FIG = Path("../5_results/figures_bootstrap")
CARPETA_FIG.mkdir(parents=True, exist_ok=True)

# =====================================================
# FUNCIÓN DE GRÁFICO
# =====================================================
def plot_roc_auc_4x3(df, nombre_modelo, output_path):
    fig, axes = plt.subplots(4, 3, figsize=(16, 10), sharey=True)
    axes = axes.flatten()

    for ax, signo in zip(axes, signos):
        df_s = df[df["signo"] == signo].sort_values("iter")
        media = df_s["roc_auc"].mean()

        ax.scatter(df_s["iter"], df_s["roc_auc"], s=12, alpha=0.5)
        ax.axhline(0.5, color="red", linestyle="--", linewidth=1)
        ax.axhline(media, color="blue", linewidth=1.5)

        ax.set_title(
            f"{signo.replace('signo_zodiacal_', '').capitalize()} (μ = {media:.3f})",
            fontsize=11
        )

        ax.set_ylim(0.3, 0.7)
        ax.set_xlabel("Iteración")
        ax.set_ylabel("ROC-AUC")

    plt.suptitle(
        f"Distribución de ROC-AUC por iteración\n({nombre_modelo} + Bootstrap + CV)",
        fontsize=15
    )

    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.savefig(output_path, dpi=300)
    plt.close()

# =====================================================
# EJECUCIÓN POR MODELO
# =====================================================
for nombre_modelo, ruta_csv in MODELOS.items():
    print(f"📊 Generando figura para {nombre_modelo}")
    df = pd.read_csv(ruta_csv)

    out_img = CARPETA_FIG / f"roc_auc_4x3_{nombre_modelo.replace(' ', '_').lower()}.png"
    plot_roc_auc_4x3(df, nombre_modelo, out_img)

    print(f"   ✅ Guardado en: {out_img}")

print("\n✅ Todas las figuras generadas")


📊 Generando figura para Random Forest
   ✅ Guardado en: ..\5_results\figures_bootstrap\roc_auc_4x3_random_forest.png
📊 Generando figura para Árbol de Decisión
   ✅ Guardado en: ..\5_results\figures_bootstrap\roc_auc_4x3_árbol_de_decisión.png
📊 Generando figura para KNN
   ✅ Guardado en: ..\5_results\figures_bootstrap\roc_auc_4x3_knn.png
📊 Generando figura para Regresión Logística
   ✅ Guardado en: ..\5_results\figures_bootstrap\roc_auc_4x3_regresión_logística.png

✅ Todas las figuras generadas
